<a href="https://colab.research.google.com/github/Maximi652/efficient-slm-architectures/blob/main/LTH_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Lottery Ticket Hypothesis for Qwen3-4B QA Task - Optimized for Colab A100
# Integrated with your specific dataset and evaluation setup

import subprocess
import sys

def install_requirements():
    """Install required packages for Colab environment."""
    requirements = [
        "transformers>=4.35.0",
        "torch>=2.0.0",
        "accelerate",
        "datasets",
        "matplotlib",
        "seaborn",
        "scikit-learn"
    ]

    for package in requirements:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])

# Uncomment to install (run once in Colab)
install_requirements()

import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM, GenerationConfig
import numpy as np
from typing import Dict, List, Tuple, Optional
import copy
import matplotlib.pyplot as plt
import seaborn as sns
import json
import os
import re
from datetime import datetime
import gc
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, f1_score

# Set up for Colab environment
os.environ["TOKENIZERS_PARALLELISM"] = "false"
plt.style.use('default')

class QwenLotteryTicketPruner:
    """
    Lottery Ticket Pruner specifically designed for Qwen3-4B QA task.
    Optimized for Colab A100 environment with QA-specific evaluation.
    """

    def __init__(self, model, pruning_rate: float = 0.2, structured: bool = False,
                 save_checkpoints: bool = True):
        self.model = model
        self.pruning_rate = pruning_rate
        self.structured = structured
        self.save_checkpoints = save_checkpoints
        self.initial_weights = {}
        self.masks = {}
        self.device = next(model.parameters()).device

        # Colab-specific paths
        self.checkpoint_dir = "/content/drive/MyDrive/lottery_tickets_qwen3_qa"
        os.makedirs(self.checkpoint_dir, exist_ok=True)

        self._store_initial_weights()
        self._print_memory_usage("After initialization")

    def _print_memory_usage(self, stage: str):
        """Print current GPU memory usage."""
        if torch.cuda.is_available():
            allocated = torch.cuda.memory_allocated() / 1024**3
            reserved = torch.cuda.memory_reserved() / 1024**3
            print(f"{stage}: GPU Memory - Allocated: {allocated:.2f}GB, Reserved: {reserved:.2f}GB")

    def _store_initial_weights(self):
        """Store initial weights with memory optimization and validation."""
        print("Storing initial weights (lottery ticket)...")

        # Only store trainable parameters from transformer layers
        # Skip embedding layers to maintain model functionality
        excluded_layers = ['embed_tokens', 'embed_positions', 'lm_head']

        stored_count = 0
        for name, param in self.model.named_parameters():
            if param.requires_grad and len(param.shape) > 1:
                # Skip embedding layers and head for QA task
                if not any(excluded in name for excluded in excluded_layers):
                    try:
                        # Store on CPU to save GPU memory
                        self.initial_weights[name] = param.data.cpu().clone()

                        # Initialize mask with proper dtype
                        mask = torch.ones_like(param.data, dtype=torch.float32)
                        self.masks[name] = mask

                        stored_count += 1
                        param_count = param.numel()
                        print(f"  Stored {name}: {param_count:,} parameters")

                    except Exception as e:
                        print(f"Error storing {name}: {e}")
                        continue

        print(f"Stored {stored_count} parameter tensors for pruning")

        # Validate masks
        self._validate_masks()

        if self.save_checkpoints:
            self.save_checkpoint("initial_state")

    def _validate_masks(self):
        """Validate that all masks are properly initialized."""
        print("Validating masks...")

        for name, mask in self.masks.items():
            if not torch.is_tensor(mask):
                print(f"Error: {name} mask is not a tensor: {type(mask)}")
                continue

            if torch.any(torch.isnan(mask)) or torch.any(torch.isinf(mask)):
                print(f"Error: {name} mask contains NaN or Inf values")
                # Reset to ones
                self.masks[name] = torch.ones_like(mask)
                continue

            total_params = mask.numel()
            active_params = (mask > 0).sum().item()
            print(f"  {name}: {active_params:,}/{total_params:,} active parameters")

    def calculate_importance_scores(self, method: str = "magnitude") -> Dict[str, torch.Tensor]:
        """Calculate importance scores for pruning."""
        importance_scores = {}

        for name, param in self.model.named_parameters():
            if name in self.masks:
                if method == "magnitude":
                    scores = torch.abs(param.data)
                elif method == "gradient" and param.grad is not None:
                    scores = torch.abs(param.grad)
                else:
                    scores = torch.abs(param.data)  # fallback to magnitude

                importance_scores[name] = scores

        return importance_scores

    def prune_global_magnitude(self):
        """Memory-efficient global magnitude pruning with better error handling."""
        print(f"Performing memory-efficient global magnitude pruning (rate: {self.pruning_rate})")

        # Step 1: Calculate statistics with validation
        total_unpruned = 0
        layer_stats = {}

        print("Counting unpruned weights per layer...")
        for name, param in self.model.named_parameters():
            if name in self.masks:
                try:
                    current_mask = self.masks[name]

                    # Validate mask
                    if current_mask.dtype != torch.bool and not torch.is_floating_point(current_mask):
                        print(f"Warning: Invalid mask dtype for {name}: {current_mask.dtype}")
                        continue

                    # Count unpruned weights
                    if torch.is_floating_point(current_mask):
                        unpruned_count = (current_mask > 0).sum().item()
                    else:
                        unpruned_count = current_mask.sum().item()

                    # Validate count
                    if not isinstance(unpruned_count, (int, float)) or unpruned_count < 0:
                        print(f"Warning: Invalid count for {name}: {unpruned_count}")
                        continue

                    if unpruned_count > 0:
                        layer_stats[name] = int(unpruned_count)
                        total_unpruned += int(unpruned_count)
                        print(f"  {name}: {unpruned_count:,} unpruned weights")

                except Exception as e:
                    print(f"Error processing layer {name}: {e}")
                    continue

        print(f"Total unpruned weights: {total_unpruned:,}")

        # Validate total
        if total_unpruned <= 0 or not isinstance(total_unpruned, int):
            print(f"Error: Invalid total_unpruned: {total_unpruned}")
            return

        if total_unpruned > 1e10:  # Sanity check for very large numbers
            print(f"Error: Suspiciously large number of weights: {total_unpruned:,}")
            return

        num_to_prune = int(total_unpruned * self.pruning_rate)
        print(f"Target: prune {num_to_prune:,} weights from {total_unpruned:,} total")

        if num_to_prune <= 0:
            print("Nothing to prune!")
            return

        # Step 2: Use percentile-based approach instead of topk
        # Sample scores to estimate global threshold
        sample_scores = []
        sample_size = min(100000, total_unpruned // 20)  # Reduced sample size for safety

        if sample_size <= 0:
            print("Sample size too small!")
            return

        print(f"Sampling {sample_size:,} weights to estimate threshold...")
        sampled_count = 0

        try:
            for name, param in self.model.named_parameters():
                if name in self.masks and name in layer_stats:
                    current_mask = self.masks[name]
                    if (current_mask > 0).sum() > 0:
                        scores = torch.abs(param.data)
                        unpruned_scores = scores[current_mask > 0]

                        # Sample from this layer
                        layer_sample_size = min(len(unpruned_scores),
                                              max(1, int(sample_size * layer_stats[name] / total_unpruned)))

                        if layer_sample_size > 0:
                            if len(unpruned_scores) > layer_sample_size:
                                indices = torch.randperm(len(unpruned_scores))[:layer_sample_size]
                                sampled = unpruned_scores[indices]
                            else:
                                sampled = unpruned_scores

                            sample_scores.append(sampled.cpu())  # Move to CPU
                            sampled_count += len(sampled)

            if not sample_scores or sampled_count == 0:
                print("No scores sampled! Using layer-wise pruning instead.")
                self._fallback_layerwise_pruning()
                return

            # Calculate threshold from samples
            print(f"Calculating threshold from {sampled_count:,} sampled weights...")
            all_samples = torch.cat(sample_scores)

            # Validate samples
            if torch.any(torch.isnan(all_samples)) or torch.any(torch.isinf(all_samples)):
                print("Warning: NaN or Inf values in samples, cleaning...")
                all_samples = all_samples[torch.isfinite(all_samples)]

            if len(all_samples) == 0:
                print("No valid samples! Falling back to layer-wise pruning.")
                self._fallback_layerwise_pruning()
                return

            percentile = self.pruning_rate * 100
            threshold = torch.quantile(all_samples, percentile / 100.0).item()

            print(f"Estimated threshold: {threshold:.6f}")

        except Exception as e:
            print(f"Error during sampling: {e}")
            print("Falling back to layer-wise pruning...")
            self._fallback_layerwise_pruning()
            return

        # Step 3: Apply pruning layer by layer
        pruned_count = 0

        try:
            for name, param in self.model.named_parameters():
                if name in self.masks:
                    current_mask = self.masks[name]
                    if (current_mask > 0).sum() > 0:
                        scores = torch.abs(param.data)

                        # Find weights to prune
                        to_prune = (scores <= threshold) & (current_mask > 0)
                        layer_pruned = to_prune.sum().item()

                        # Apply pruning
                        current_mask[to_prune] = 0

                        pruned_count += layer_pruned

                        if layer_pruned > 0:
                            print(f"  {name}: pruned {layer_pruned:,} weights")

            print(f"Total pruned: {pruned_count:,} weights (threshold: {threshold:.6f})")

        except Exception as e:
            print(f"Error during pruning application: {e}")

        finally:
            # Clean up memory
            if 'sample_scores' in locals():
                del sample_scores
            if 'all_samples' in locals():
                del all_samples
            torch.cuda.empty_cache()

    def _fallback_layerwise_pruning(self):
        """Fallback to simple layer-wise pruning if global pruning fails."""
        print("Using fallback layer-wise pruning...")

        pruned_count = 0
        for name, param in self.model.named_parameters():
            if name in self.masks:
                current_mask = self.masks[name]
                active_weights = (current_mask > 0).sum().item()

                if active_weights > 0:
                    scores = torch.abs(param.data)
                    unpruned_scores = scores[current_mask > 0]

                    layer_prune_count = int(active_weights * self.pruning_rate)
                    if layer_prune_count > 0:
                        threshold = torch.topk(unpruned_scores, layer_prune_count, largest=False)[0][-1]

                        to_prune = (scores <= threshold) & (current_mask > 0)
                        current_mask[to_prune] = 0

                        actual_pruned = to_prune.sum().item()
                        pruned_count += actual_pruned

                        print(f"  {name}: pruned {actual_pruned:,}/{active_weights:,} weights")

        print(f"Fallback pruning completed: {pruned_count:,} weights pruned")

    def apply_masks(self):
        """Apply current masks to model parameters."""
        for name, param in self.model.named_parameters():
            if name in self.masks:
                param.data *= self.masks[name]

    def reset_to_lottery_ticket(self):
        """Reset to initial weights with current masks."""
        print("Resetting to lottery ticket...")

        for name, param in self.model.named_parameters():
            if name in self.initial_weights:
                initial_weight = self.initial_weights[name].to(self.device)
                param.data.copy_(initial_weight)
                param.data *= self.masks[name]
                del initial_weight

        torch.cuda.empty_cache()
        self._print_memory_usage("After lottery ticket reset")

    def get_sparsity_stats(self) -> Dict[str, float]:
        """Get detailed sparsity statistics."""
        stats = {'layers': {}}
        total_params = 0
        total_pruned = 0

        for name, mask in self.masks.items():
            layer_total = mask.numel()
            layer_pruned = (mask == 0).sum().item()
            layer_sparsity = layer_pruned / layer_total

            stats['layers'][name] = {
                'sparsity': layer_sparsity,
                'remaining': layer_total - layer_pruned,
                'total': layer_total
            }

            total_params += layer_total
            total_pruned += layer_pruned

        stats['overall_sparsity'] = total_pruned / total_params
        stats['total_params'] = total_params
        stats['remaining_params'] = total_params - total_pruned

        return stats

    def save_checkpoint(self, checkpoint_name: str):
        """Save current state to Google Drive."""
        if not self.save_checkpoints:
            return

        checkpoint_path = os.path.join(self.checkpoint_dir, f"{checkpoint_name}.pt")

        checkpoint = {
            'masks': {name: mask.cpu() for name, mask in self.masks.items()},
            'sparsity_stats': self.get_sparsity_stats(),
            'pruning_rate': self.pruning_rate,
            'timestamp': datetime.now().isoformat()
        }

        torch.save(checkpoint, checkpoint_path)
        print(f"Checkpoint saved: {checkpoint_path}")

class QwenQALotteryExperiment:
    """
    Complete lottery ticket experiment for Qwen3-4B QA task.
    Integrated with your specific dataset and evaluation metrics.
    """

    def __init__(self, model_path: str, input_json: str):
        self.model_path = model_path
        self.input_json = input_json
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print(f"Using device: {self.device}")

        # Load your specific model and tokenizer
        self.setup_model_and_tokenizer()
        self.load_dataset()

    def setup_model_and_tokenizer(self):
        """Load Qwen3-4B model and tokenizer with your exact configuration."""
        print(f"Loading model from {self.model_path}...")

        # Load tokenizer with your exact settings
        self.tokenizer = AutoTokenizer.from_pretrained(
            self.model_path,
            trust_remote_code=True
        )

        # Your exact tokenizer configuration
        self.tokenizer.padding_side = 'left'
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        # Load model with memory optimization
        self.model = AutoModelForCausalLM.from_pretrained(
            self.model_path,
            torch_dtype=torch.float16,  # Use FP16 for memory efficiency
            device_map="auto",
            trust_remote_code=True
        )

        # Your generation configuration
        self.gen_conf = GenerationConfig(
            max_new_tokens=200,
            do_sample=False
        )

        print(f"Model loaded successfully. Parameters: ~4B")
        self._print_memory_usage("After model loading")

    def _print_memory_usage(self, stage: str):
        """Print GPU memory usage."""
        if torch.cuda.is_available():
            allocated = torch.cuda.memory_allocated() / 1024**3
            reserved = torch.cuda.memory_reserved() / 1024**3
            print(f"{stage}: GPU Memory - Allocated: {allocated:.2f}GB, Reserved: {reserved:.2f}GB")

    def load_dataset(self):
        """Load your specific QA dataset."""
        print(f"Loading dataset from {self.input_json}...")

        with open(self.input_json, 'r', encoding='utf-8') as f:
            data = json.load(f)
            self.questions = data['questions']

        print(f"Loaded {len(self.questions)} questions")

        # Split into train/eval for lottery ticket experiments
        split_idx = int(len(self.questions) * 0.8)
        self.train_questions = self.questions[:split_idx]
        self.eval_questions = self.questions[split_idx:]

        print(f"Train: {len(self.train_questions)}, Eval: {len(self.eval_questions)}")

    def build_messages(self, qtext, snippets, qtype, mode="exact"):
        """Your exact message building function."""
        system_msg = {"role":"system","content":"/no_think"}
        ctx = "\n".join(s["text"] for s in snippets[:2])

        if mode == "exact":
            if qtype == "yesno":
                content = f"Question: {qtext}\nContext:\n{ctx}\nAnswer only 'yes' or 'no', in English, no extras."
            elif qtype == "factoid":
                content = f"Question: {qtext}\nContext:\n{ctx}\nProvide up to 5 keywords, comma-separated, in English, no commentary."
            elif qtype == "list":
                content = f"Question: {qtext}\nContext:\n{ctx}\nProvide a comma-separated list of relevant items, in English, no filler words."
            else:
                content = f"Question: {qtext}\nContext:\n{ctx}\nProvide a brief answer in English."
        else:  # ideal
            if qtype == "yesno":
                content = f"Question: {qtext}\nContext:\n{ctx}\nProvide one-sentence ideal answer in English starting with 'Yes,' or 'No,'."
            else:
                content = f"Question: {qtext}\nContext:\n{ctx}\nProvide an ideal answer in English (one paragraph, max 200 words, full sentences)."

        user_msg = {"role":"user","content":content}
        return [system_msg, user_msg]

    def clean_exact(self, text, qtype):
        """Your exact cleaning function."""
        txt = text.strip()
        txt = re.sub(r'<\/think>','', txt)
        txt = re.sub(r'\s*(Okay\.?|etc\.?|usw\.?|\.\.\.)$', '', txt, flags=re.IGNORECASE)

        if qtype == "yesno":
            return "yes" if txt.lower().startswith("yes") else "no"
        if qtype in ("factoid","list"):
            items = [i.strip() for i in txt.split(",") if i.strip()]
            return items
        return txt

    def clean_ideal(self, text, qtype):
        """Your exact cleaning function for ideal answers."""
        txt = text.strip()
        txt = re.sub(r'<\/think>','', txt)
        txt = re.sub(r'\s*(Okay\.?|etc\.?|usw\.?|\.\.\.)$', '', txt, flags=re.IGNORECASE)

        sentences = re.split(r'(?<=[.!?])\s+', txt)
        if qtype == "yesno":
            return sentences[0].strip()

        total = 0
        out = []
        for sent in sentences:
            length = len(sent.split())
            if total + length <= 200:
                out.append(sent)
                total += length
            else:
                break
        return " ".join(out).strip()

    def evaluate_qa_performance(self, questions_subset, batch_size=4):
        """Evaluate QA performance using your exact evaluation logic."""
        self.model.eval()
        results = []

        with torch.no_grad():
            for i in tqdm(range(0, len(questions_subset), batch_size), desc='Evaluating'):
                batch = questions_subset[i:i+batch_size]

                # Exact answers
                msgs_ex = [self.build_messages(q['body'], q.get('snippets',[]), q['type'], mode='exact') for q in batch]
                texts_ex = [self.tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=True, enable_thinking=False) for m in msgs_ex]
                inputs_ex = self.tokenizer(texts_ex, return_tensors='pt', padding=True, truncation=True).to(self.model.device)

                out_ex = self.model.generate(**inputs_ex, generation_config=self.gen_conf)

                dec_ex = []
                for idx, q in enumerate(batch):
                    start = inputs_ex['input_ids'].shape[1]
                    ids = out_ex[idx][start:].tolist()
                    text = self.tokenizer.decode(ids, skip_special_tokens=True)
                    dec_ex.append(self.clean_exact(text, q['type']))

                # Store results
                for idx, q in enumerate(batch):
                    results.append({
                        'id': q['id'],
                        'type': q['type'],
                        'exact_answer': q.get('exact_answer'),
                        'exact_prediction': dec_ex[idx],
                        'question': q['body']
                    })

        return results

    def calculate_qa_metrics(self, results):
        """Calculate QA-specific metrics."""
        metrics = {'overall': {}, 'by_type': {}}

        # Group by question type
        by_type = {}
        for result in results:
            qtype = result['type']
            if qtype not in by_type:
                by_type[qtype] = {'correct': 0, 'total': 0}

            by_type[qtype]['total'] += 1

            # Check correctness based on question type
            pred = result['exact_prediction']
            truth = result['exact_answer']

            if qtype == "yesno":
                if isinstance(pred, str) and isinstance(truth, str):
                    correct = pred.lower().strip() == truth.lower().strip()
                else:
                    correct = str(pred).lower() == str(truth).lower()
            elif qtype in ["factoid", "list"]:
                if isinstance(pred, list) and isinstance(truth, list):
                    # Calculate overlap for lists
                    pred_set = set(str(x).lower().strip() for x in pred)
                    truth_set = set(str(x).lower().strip() for x in truth)
                    correct = len(pred_set & truth_set) > 0  # At least one match
                else:
                    correct = False
            else:
                # For other types, simple string matching
                correct = str(pred).lower().strip() == str(truth).lower().strip()

            if correct:
                by_type[qtype]['correct'] += 1

        # Calculate metrics by type
        total_correct = 0
        total_questions = 0

        for qtype, stats in by_type.items():
            accuracy = stats['correct'] / stats['total'] if stats['total'] > 0 else 0
            metrics['by_type'][qtype] = {
                'accuracy': accuracy,
                'correct': stats['correct'],
                'total': stats['total']
            }
            total_correct += stats['correct']
            total_questions += stats['total']

        # Overall metrics
        metrics['overall'] = {
            'accuracy': total_correct / total_questions if total_questions > 0 else 0,
            'correct': total_correct,
            'total': total_questions
        }

        return metrics

    def run_lottery_ticket_qa_experiment(self,
                                       num_iterations: int = 5,
                                       pruning_rate: float = 0.15,
                                       eval_batch_size: int = 2):  # Reduced batch size
        """
        Run lottery ticket experiment specifically for QA task.
        """
        # Initialize pruner
        pruner = QwenLotteryTicketPruner(self.model, pruning_rate=pruning_rate)

        # Results tracking
        results = []

        print("\n" + "="*60)
        print("STARTING QA LOTTERY TICKET EXPERIMENT")
        print("="*60)

        for iteration in range(num_iterations):
            print(f"\n🎯 ITERATION {iteration + 1}/{num_iterations}")
            print("-" * 40)

            # Clear cache before each iteration
            torch.cuda.empty_cache()
            gc.collect()

            # Reset to lottery ticket (except first iteration)
            if iteration > 0:
                pruner.reset_to_lottery_ticket()

            # Apply current masks
            pruner.apply_masks()

            # Evaluate on smaller subset for memory efficiency
            eval_subset = self.eval_questions[:min(50, len(self.eval_questions))]  # Reduced from 100
            print(f"Evaluating on {len(eval_subset)} questions...")

            qa_results = self.evaluate_qa_performance(eval_subset, batch_size=eval_batch_size)
            metrics = self.calculate_qa_metrics(qa_results)
            sparsity_stats = pruner.get_sparsity_stats()

            # Store results
            result = {
                'iteration': iteration + 1,
                'overall_accuracy': metrics['overall']['accuracy'],
                'yesno_accuracy': metrics['by_type'].get('yesno', {}).get('accuracy', 0),
                'factoid_accuracy': metrics['by_type'].get('factoid', {}).get('accuracy', 0),
                'list_accuracy': metrics['by_type'].get('list', {}).get('accuracy', 0),
                'sparsity': sparsity_stats['overall_sparsity'],
                'remaining_params': sparsity_stats['remaining_params'],
                'total_params': sparsity_stats['total_params'],
                'detailed_metrics': metrics
            }
            results.append(result)

            # Print results
            print(f"📈 Results:")
            print(f"  Overall Accuracy: {result['overall_accuracy']:.2%}")
            print(f"  Yes/No Accuracy: {result['yesno_accuracy']:.2%}")
            print(f"  Factoid Accuracy: {result['factoid_accuracy']:.2%}")
            print(f"  List Accuracy: {result['list_accuracy']:.2%}")
            print(f"  Sparsity: {result['sparsity']:.2%}")
            print(f"  Remaining Params: {result['remaining_params']:,}")

            # Save checkpoint
            pruner.save_checkpoint(f"qa_iteration_{iteration + 1}")

            # Memory cleanup before pruning
            torch.cuda.empty_cache()
            gc.collect()

            # Pruning phase (except last iteration)
            if iteration < num_iterations - 1:
                print("✂️ Pruning phase...")
                pruner.prune_global_magnitude()

                # Additional cleanup after pruning
                torch.cuda.empty_cache()
                gc.collect()

        print("\n" + "="*60)
        print("QA EXPERIMENT COMPLETED!")
        print("="*60)

        return self.model, pruner, results

    def plot_qa_results(self, results):
        """Create QA-specific visualizations."""
        fig, axes = plt.subplots(2, 2, figsize=(15, 10))

        iterations = [r['iteration'] for r in results]

        # Plot 1: Accuracy by question type
        axes[0,0].plot(iterations, [r['overall_accuracy'] for r in results], 'o-', label='Overall', linewidth=2)
        axes[0,0].plot(iterations, [r['yesno_accuracy'] for r in results], 's-', label='Yes/No')
        axes[0,0].plot(iterations, [r['factoid_accuracy'] for r in results], '^-', label='Factoid')
        axes[0,0].plot(iterations, [r['list_accuracy'] for r in results], 'd-', label='List')
        axes[0,0].set_xlabel('Iteration')
        axes[0,0].set_ylabel('Accuracy')
        axes[0,0].set_title('QA Accuracy by Question Type')
        axes[0,0].legend()
        axes[0,0].grid(True, alpha=0.3)

        # Plot 2: Accuracy vs Sparsity
        axes[0,1].scatter([r['sparsity'] for r in results], [r['overall_accuracy'] for r in results],
                         c=iterations, cmap='viridis', s=100)
        axes[0,1].set_xlabel('Sparsity')
        axes[0,1].set_ylabel('Overall Accuracy')
        axes[0,1].set_title('Accuracy vs Sparsity Trade-off')
        axes[0,1].grid(True, alpha=0.3)

        # Plot 3: Parameter count
        axes[1,0].bar(iterations, [r['remaining_params'] for r in results], alpha=0.7, color='skyblue')
        axes[1,0].set_xlabel('Iteration')
        axes[1,0].set_ylabel('Remaining Parameters')
        axes[1,0].set_title('Model Size Reduction')
        axes[1,0].grid(True, alpha=0.3)

        # Plot 4: Sparsity progression
        axes[1,1].plot(iterations, [r['sparsity'] * 100 for r in results], 'go-', linewidth=3, markersize=8)
        axes[1,1].set_xlabel('Iteration')
        axes[1,1].set_ylabel('Sparsity (%)')
        axes[1,1].set_title('Sparsity Progression')
        axes[1,1].grid(True, alpha=0.3)

        plt.tight_layout()
        plt.savefig('/content/drive/MyDrive/lottery_tickets_qwen3_qa/qa_results_plot.png',
                   dpi=300, bbox_inches='tight')
        plt.show()

        return fig

# Main execution function for your QA task
def run_qwen3_qa_lottery_experiment():
    """
    Main function to run lottery ticket experiment on your Qwen3-4B QA task.
    Uses your exact model, dataset, and evaluation setup.
    """
    # Your exact file paths
    MODEL_PATH = "/content/drive/MyDrive/Colab Notebooks/Qwen3-4B"
    INPUT_JSON = "/content/drive/MyDrive/Colab Notebooks/12B_combined_golden.json"

    # Check GPU and set memory optimization
    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name()
        gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1024**3
        print(f"🚀 GPU Detected: {gpu_name}")
        print(f"💾 GPU Memory: {gpu_memory:.1f} GB")

        # Enable memory optimization
        os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True

        if gpu_memory < 20:
            print("⚠️ Warning: Less than 20GB GPU memory. Using ultra-conservative settings.")
            eval_batch_size = 1
            pruning_rate = 0.05  # Lower pruning rate
        else:
            eval_batch_size = 2
            pruning_rate = 0.1  # More conservative than 0.3
    else:
        print("❌ No GPU detected! This experiment requires CUDA.")
        return None

    # Clear any existing cached memory
    torch.cuda.empty_cache()

    try:
        # Create experiment
        experiment = QwenQALotteryExperiment(MODEL_PATH, INPUT_JSON)

        # Run lottery ticket experiment with conservative settings
        model, pruner, results = experiment.run_lottery_ticket_qa_experiment(
            num_iterations=5,  # Reduced from 4
            pruning_rate=pruning_rate,
            eval_batch_size=eval_batch_size
        )

        # Create visualizations
        fig = experiment.plot_qa_results(results)

        # Print final summary
        print("\n📊 FINAL QA LOTTERY TICKET SUMMARY")
        print("-" * 50)
        for i, result in enumerate(results):
            print(f"Iteration {i+1}: "
                  f"Sparsity={result['sparsity']:.1%}, "
                  f"Overall Acc={result['overall_accuracy']:.2%}, "
                  f"Yes/No={result['yesno_accuracy']:.2%}, "
                  f"Factoid={result['factoid_accuracy']:.2%}")

        # Save detailed results
        results_path = "/content/drive/MyDrive/lottery_tickets_qwen3_qa/detailed_results.json"
        with open(results_path, 'w') as f:
            json.dump(results, f, indent=2)
        print(f"\n💾 Detailed results saved to: {results_path}")

        return model, pruner, results, fig

    except torch.cuda.OutOfMemoryError as e:
        print(f"💥 CUDA OutOfMemoryError: {e}")
        print("\n🔧 TROUBLESHOOTING TIPS:")
        print("1. Restart runtime to clear all memory")
        print("2. Try smaller batch size or fewer evaluation questions")
        print("3. Use gradient checkpointing")
        print("4. Consider using a smaller model for testing")

        # Clear memory
        torch.cuda.empty_cache()
        gc.collect()

        return None

# Usage instructions for Colab:
print("""
🚀 QWEN3-4B QA LOTTERY TICKET EXPERIMENT 🚀
""")

# Uncomment to run the experiment
from google.colab import drive
drive.mount('/content/drive')

results = run_qwen3_qa_lottery_experiment()